# Sudoku Knowledge Representation & Inference
**IT5005 | Assignment and Project Group 22**

The implementations are in `sudoku_solver.py`. This notebook records the encodings,
experiments and checks; `sudoku_app.py` imports the same solver. Place the three
submitted files beside the original `utils.py`, `logic_.py` and `puzzles.json` to run them.


In [1]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
    backward_proof,
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [2]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), '.') for c in range(1, n + 1)])

5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
['.', 3, '.', '.', '.', '.', '.', '.', '.']
[2, '.', '.', '.', '.', 7, 8, 6, '.']
[5, 8, '.', 2, 6, '.', '.', 3, '.']
[7, 5, '.', '.', '.', '.', '.', 8, '.']
['.', '.', '.', '.', 7, '.', 5, '.', 4]
['.', '.', '.', 5, 3, '.', '.', 9, 6]
['.', 1, 2, '.', '.', 9, '.', '.', '.']
[6, 4, '.', '.', 5, 8, 9, '.', '.']
['.', '.', '.', '.', 2, 3, '.', '.', '.']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [3]:
print(atom('Is', 3, 2, 4), atom('Not', 3, 2, 4))

Is3_2_4 Not3_2_4


## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [4]:
general_kb = build_general_kb(n, box_h, box_w, givens)
definite_kb = build_definite_kb(n, box_h, box_w, givens)
assert isinstance(general_kb, PropKB)
assert isinstance(definite_kb, PropDefiniteKB)
assert all(is_definite_clause(c) for c in definite_kb.clauses)
print('General KB:', len(general_kb.clauses), 'clauses')
print('Definite KB:', len(definite_kb.clauses), 'clauses')
print('General symbols:', len(prop_symbols(associate('&', general_kb.clauses))))
print('Example elimination:', expr('Is1_1_5 ==> Not1_2_5'))


General KB: 10317 clauses
Definite KB: 21171 clauses
General symbols: 729
Example elimination: (Is1_1_5 ==> Not1_2_5)


### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Attempt this in the code cell below: it is provided commented out, because both are sound and complete on `build_general_kb`'s output but neither scales to the full grid, and it is expected to hang or take an impractically long time. Uncomment a few lines at a time and give each at most about 30 seconds; use Kernel > Interrupt if it hasn't returned by then, and note what you observed.

Explain in your own words: what specifically makes `pl_resolution`'s cost grow out of control here, and separately, what makes `tt_entails`'s cost grow out of control? A simple complexity argument for each is the expected answer.

**Use your observations in Conceptual Question 2 below.**


In [5]:
# Run each supplied algorithm in an isolated process so a 30-second
# experiment cannot leave the notebook kernel stuck or keep consuming memory.
import subprocess
import sys

experiment_code = r"""
import json, sys, time
from sudoku_solver import atom, build_general_kb
from logic_ import pl_resolution, tt_entails, associate
raw = json.load(open('puzzles.json'))
p = raw['puzzles'][0]
givens = {tuple(map(int, k.split('_'))): v for k, v in p['givens'].items()}
kb = build_general_kb(raw['n'], raw['box_h'], raw['box_w'], givens)
cell = next((r,c) for r in range(1,raw['n']+1) for c in range(1,raw['n']+1)
            if (r,c) not in givens)
q = atom('Is', *cell, 1)
print(f'Query: {q}; clauses: {len(kb.clauses)}', flush=True)
t = time.perf_counter()
answer = (pl_resolution(kb, q) if sys.argv[1] == 'resolution'
          else tt_entails(associate('&', kb.clauses), q))
print(f'Result: {answer}; seconds: {time.perf_counter()-t:.3f}', flush=True)
"""
observations = {}
for method in ['resolution', 'truth_table']:
    start = time.perf_counter()
    try:
        completed = subprocess.run([sys.executable, '-c', experiment_code, method],
                                   capture_output=True, text=True, timeout=30)
        status = 'returned' if completed.returncode == 0 else 'error'
        detail = completed.stdout + completed.stderr
    except subprocess.TimeoutExpired as error:
        status = 'timeout after 30 seconds; child process terminated'
        detail = error.stdout or b''
        if isinstance(detail, bytes):
            detail = detail.decode('utf-8', errors='replace')
    observations[method] = status
    print(f'{method}: {status} (wall time {time.perf_counter()-start:.2f}s)')
    print(detail.strip())

# A small control confirms that both library calls work when the KB is tiny.
tiny = build_general_kb(1, 1, 1, {})
assert pl_resolution(tiny, atom('Is', 1, 1, 1))
assert tt_entails(associate('&', tiny.clauses), atom('Is', 1, 1, 1))
print('1x1 control: both algorithms returned True.')


resolution: timeout after 30 seconds; child process terminated (wall time 30.18s)
Query: Is1_1_1; clauses: 10317


truth_table: timeout after 30 seconds; child process terminated (wall time 30.02s)
Query: Is1_1_1; clauses: 10317
1x1 control: both algorithms returned True.


### Observation notes

Both full-size calls timed out at the 30-second limit without returning a verdict.
The query was `Is1_1_1`: the first initially empty cell and candidate 1, chosen
without consulting the answer key. The child processes were terminated after the
limit. Both algorithms returned True on the 1x1 control, confirming that the calls
were valid. A timeout is not evidence that the query is false; Question 2 explains
why the full-size representations are expensive for these routines.

**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [6]:
t0 = time.perf_counter()
solved = solve_full_grid_fc(n, box_h, box_w, givens)
fc_time = time.perf_counter() - t0
t0 = time.perf_counter()
solved_bc = solve_full_grid_bc(n, box_h, box_w, givens)
bc_time = time.perf_counter() - t0
assert solved == solved_bc == puzzle['solution']
print('Solved grid (both algorithms):')
for r in range(1, n + 1):
    print(' '.join(str(solved[r,c]) for c in range(1, n + 1)))
print(f'FC: {fc_time:.3f}s; BC: {bc_time:.3f}s; FC/BC: {fc_time/bc_time:.2f}')


Solved grid (both algorithms):
1 3 6 9 8 5 4 7 2
2 9 4 3 1 7 8 6 5
5 8 7 2 6 4 1 3 9
7 5 1 4 9 6 2 8 3
3 6 9 8 7 2 5 1 4
4 2 8 5 3 1 7 9 6
8 1 2 6 4 9 3 5 7
6 4 3 7 5 8 9 2 1
9 7 5 1 2 3 6 4 8
FC: 17.045s; BC: 1.056s; FC/BC: 16.14


### Verification across the complete puzzle pool

In [7]:
# The answer key is used only by assertions, never by an inference function.
results = []
for i, item in enumerate(puzzle_pool, 1):
    timings = {}
    for name, solver in [('FC', solve_full_grid_fc), ('BC', solve_full_grid_bc)]:
        t0 = time.perf_counter()
        grid = solver(n, box_h, box_w, item['givens'])
        timings[name] = time.perf_counter() - t0
        assert grid == item['solution']
    kb = build_definite_kb(n, box_h, box_w, item['givens'])
    checks = 0
    for (r,c), correct in item['solution'].items():
        for candidate in range(1,n+1):
            q = atom('Is', r, c, candidate)
            expected = candidate == correct
            assert pl_bc_entails(kb, q) == expected
            assert pl_fc_entails(kb, q) == expected
            checks += 1
    results.append((i, item['given_count'], timings['FC'], timings['BC'], checks))
print('Puzzle | Givens | FC seconds | BC seconds | FC/BC candidate checks')
for i, count, fc, bc, checks in results:
    print(f'{i:6} | {count:6} | {fc:10.3f} | {bc:10.3f} | {checks:4} passed')
print('Total:', sum(row[4] for row in results), 'candidate pairs checked with both algorithms.')

# Regression: a cyclic dependency has a finite proof through an alternative rule.
cycle = PropDefiniteKB()
for clause in ['B ==> A', 'A ==> B', 'C ==> A', 'C']:
    cycle.tell(expr(clause))
assert pl_bc_entails(cycle, expr('B'))
cycle.retract(expr('C'))
assert not pl_bc_entails(cycle, expr('B'))
print('Cycle, alternative proof, and cache invalidation checks passed.')


Puzzle | Givens | FC seconds | BC seconds | FC/BC candidate checks
     1 |     30 |     17.091 |      1.053 |  729 passed
     2 |     33 |     16.998 |      1.093 |  729 passed
     3 |     36 |     16.888 |      1.065 |  729 passed
     4 |     39 |     16.788 |      1.084 |  729 passed
     5 |     42 |     16.905 |      1.115 |  729 passed
Total: 3645 candidate pairs checked with both algorithms.
Cycle, alternative proof, and cache invalidation checks passed.


## Part B: Conceptual Questions

### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


**(a) General representation.** Write $I_{rcv}$ for `Is(r,c,v)`.
For each cell I add the clause $I_{rc1}\lor\cdots\lor I_{rcn}$.
For each pair $v<w$, I add $\neg I_{rcv}\lor\neg I_{rcw}$.
Together these say that a cell has exactly one value.

Two distinct cells are peers if they share a row, column or box. For every unordered
peer pair $(r,c),(s,t)$ and every value $v$, I add
$\neg I_{rcv}\lor\neg I_{stv}$. Thus a digit cannot repeat in a row, column or box.
Every unit has $n$ cells taking distinct values from a set of $n$ values, so each digit
also occurs at least once in that unit; extra unit-level existence clauses are unnecessary.
A given $(r,c)=v$ is the unit clause $I_{rcv}$.

All these sentences are already CNF and are passed to `PropKB.tell`.
Only the `Is` family is needed: logical negation expresses exclusion directly.
The peer set removes duplicates where two cells share both a row and a box.
For a 9x9 board, each cell has 20 peers. With $g$ givens, the encoding has
$81+81\binom{9}{2}+(81\cdot20/2)9+g=10,287+g$ clauses and 729 atoms.

**(b) Definite representation.** I use a second positive atom $N_{rcv}$, written
`Not(r,c,v)`, to record an established exclusion. Despite its name, this is an atom,
not the formula $\neg I_{rcv}$. Every implication below has only positive premises
and exactly one positive conclusion, so `PropDefiniteKB.tell` accepts it.

1. A given is the fact $I_{rcv}$.
2. A value excludes other values in the same cell:
   $I_{rcv}\Rightarrow N_{rcw}$ for every $w\ne v$.
3. A value excludes itself from every peer:
   $I_{rcv}\Rightarrow N_{stv}$ whenever $(s,t)$ is a peer of $(r,c)$.
   This covers the row, column and box constraints.
4. If every other value in a cell is excluded, the remaining value is forced:
   $\bigwedge_{w\ne v}N_{rcw}\Rightarrow I_{rcv}$.

The last rule uses the Sudoku requirement that each cell must have a value, without
placing a multi-positive disjunction in the Horn KB. For $n=1$ it becomes an
unconditional fact. The 9x9 rule count is
$729(8+20+1)+g=21,141+g$.

This distinction matters: the Horn KB is a sound *deduction system* for Sudoku,
not a logically equivalent replacement for the full CNF. Its atoms do not enforce
`Not` as the Boolean complement of `Is`, and an unresolved cell need not receive
any `Is` atom in the least model. Therefore complete Horn inference can still stop
short of solving a valid Sudoku. My solvers report that case instead of guessing.
All five supplied puzzles are checked above using only these elimination and
last-candidate rules.

### 2. Theoretical Completeness vs. Computational Tractability

Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


I would not use either supplied algorithm as the default 9x9 solver.
Completeness answers whether an unlimited run finds the right answer; it does not
make that run affordable. The bounded experiments above give the practical result
for this implementation and machine, rather than an assumed runtime.

**Truth-table model checking.** There are $9^3=729$ Boolean `Is` atoms, so the
truth table has $2^{729}\approx2.82\times10^{219}$ assignments. Most are not legal
Sudoku boards, but `tt_check_all` still branches over the Boolean symbols and only
evaluates the KB at a leaf. In the worst case its time is
$O(2^m(L+|q|))$, where $m$ is the number of symbols and $L$ is the KB's total
literal count. A false query may stop at a countermodel; an entailed query requires
the exhaustive check. The implementation traverses depth first, so it does not
store the entire truth table, but its recursion and copied models/symbol lists
still use polynomial auxiliary space (up to quadratic in $m$ here).

**Resolution.** `pl_resolution` first adds the negated query. At each round it
materializes all pairs of current clauses and resolves them, including pairs with
no useful complementary literals. The first puzzle has 10,317 clauses; after
adding the query, the first round alone contains
$\binom{10,318}{2}=53,225,403$ pairs.
That costs quadratic space just for the pair list, before accounting for new
resolvents and their literal comparisons. A round with $C$ clauses has
$\Theta(C^2)$ pairs; the clause set can itself grow exponentially in $m$
(up to $3^m$ distinct non-tautological clauses in a canonical representation).
The supplied code also retains redundant work across rounds. This is a different
bottleneck from enumerating truth assignments.

For the supplied puzzles, the Horn representation lets both chaining methods use
much more focused deductions. This does not imply that modern SAT solvers are
impractical: they use substantially stronger search and propagation machinery
than these two textbook routines.

### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


The algorithm treats alternative rules as **OR** choices and the premises of a
single rule as an **AND**. I index rules by their conclusion. Facts and proved
atoms are reusable; a branch that encounters a cycle is not itself a proof.

```text
BC-ENTAILS(KB, query):
    if KB contents changed: rebuild fact/rule indexes and clear cached answers
    if query is known: return True
    if query has a completed negative result: return False
    repeat:
        old_size = number of known atoms
        expanded = empty set
        if PROVE(query, expanded): return True
        if number of known atoms == old_size:
            remember query as not entailed
            return False

PROVE(goal, expanded):
    if goal is known: return True
    if goal is expanded this pass or has a completed negative result: return False
    add goal to expanded
    for each rule (p1 AND ... AND pk => goal) indexed under goal:
        success = True
        for each premise pi:
            if not PROVE(pi, expanded):
                success = False
                break
        if success:
            add goal to known
            record this rule's premises as the proof of goal
            return True
    return False
```

In Python, `PROVE` yields subgoals to an explicit stack of suspended generators.
This has the recursive structure above without relying on Python's recursion
limit. It also records actual successful rule applications for the app's tutor mode.

The main correctness trap is caching failure too early. Consider $B\Rightarrow A$,
$A\Rightarrow B$, $C\Rightarrow A$, with fact $C$. During a search for $A$, the
first route through $B$ revisits $A$ and is cut. That does not make $B$ false:
$C$ proves $A$, which then proves $B$. My per-pass `expanded` set cuts repeated
expansion, and another pass revisits unresolved goals whenever new facts were
proved. Only a pass with no additions permits a negative answer for the root query.
The notebook's cycle test also retracts $C$ to check that stale proofs are discarded.

Soundness follows by induction: the initial known atoms are facts, and each new
atom has a rule whose premises were already proved. For completeness, consider a
no-progress pass. The known set is fixed throughout that pass. Every unproved goal
visited has each candidate rule rejected at an unproved premise. No finite proof
tree can escape that closed set of failed dependencies: choosing a failed goal of
minimum proof height would require an even smaller failed premise. Hence the root
has no Horn proof. This justifies the final False, unlike a single cycle cut.

Each pass expands a finite goal at most once and considers finitely many rules.
An unsuccessful nonfinal pass adds at least one new atom. A finite propositional
KB has only finitely many atoms, so only finitely many such passes are possible.
Pure cycles without a supporting fact terminate with False. The method is complete
for the encoded definite KB; it cannot supply Sudoku rules that were never encoded.

### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


I would encode **Naked Pairs**. Let $a$ and $b$ be two distinct cells in one
row, column or box, and let $x\ne y$ be two digits. Define the positive conjunction

$$E=\left(\bigwedge_{v\notin\{x,y\}}N_{a,v}\right)
      \land\left(\bigwedge_{v\notin\{x,y\}}N_{b,v}\right).$$

For every other cell $d$ in that same unit, add two separate definite clauses:

$$E\Rightarrow N_{d,x},\qquad E\Rightarrow N_{d,y}.$$

For example, in row 1, suppose columns 2 and 7 have both excluded 1, 2, 4, 5,
6, 8 and 9. Their only possible values are 3 and 7. The two cells must occupy
those two values in some order, so every other cell in row 1 excludes both 3 and 7.
The premises are the fourteen corresponding `Not` atoms, and each conclusion is
one `Not` atom. No disjunction or test for an absent fact is needed.

The argument uses the Sudoku assumptions that each cell has one value and that
values do not repeat in a unit. Under a consistent puzzle, restricting both cells
to $\{x,y\}$ is sufficient; I do not need to infer their candidates from failure
to prove some `Is` atom. Explicit exclusions are what make the rule monotonic.

These rules strengthen propagation but substantially enlarge a naive grounded KB.
Across $3n$ units, there are $\binom n2$ cell pairs, $\binom n2$ digit pairs and
$2(n-2)$ conclusions per combination. For $n=9$, that is 489,888 rules before
removing duplicates, each with 14 premises. Generating relevant instances on
demand would avoid much of that cost. Naked Pairs still does not make the system
a complete solver for arbitrary Sudoku: further patterns or search may be needed.
I leave this extension out of the timed base solver so the measurements correspond
to the requested elimination and last-candidate encoding.

### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


**(a)** Suppose a KB contains 100,000 rules about unrelated cells or problems,
but a query follows from a fact through three rules. With a conclusion index,
backward chaining explores that small dependency set. Forward chaining may process
many irrelevant consequences before reaching the query. This advantage assumes
that the index is available or reused: constructing it from scratch still requires
reading the whole KB. A query that is already a fact is an even simpler case.

**(b)** If the query depends on most of the KB, backward chaining loses that
selectivity. Dense cycles and many unsuccessful alternatives add repeated proof
attempts. Asking for every cell also makes the union of relevant dependencies
large. A forward-chaining implementation that computes the closure once and reuses
it can then be faster than repeatedly starting goal-directed searches.

For the first puzzle, the full-pool run measured 17.091 s for FC and 1.053 s
for BC, about 16.2 times faster for this BC implementation. Across the five
puzzles, FC took 16.788-17.091 s and BC took 1.053-1.115 s. The output table
above records each measurement.
Each timed call starts with a fresh KB, includes KB construction, and tries values
in ascending order. The order of the two algorithms is FC then BC; these are single
wall-clock measurements, not a controlled multi-run benchmark.

There are two implementation details behind the comparison. The assignment's
`pl_fc_entails` is called separately for each candidate, so it rebuilds counters and
an agenda each time. I provide an indexed `PropDefiniteKB` subclass for premise
lookup, while leaving that library algorithm unchanged. My backward chainer keeps
proved atoms and completed negative answers within one KB, allowing later queries
in the same solve to reuse them. Its first query also builds the conclusion index.
Thus a timing advantage for BC here is evidence about these particular routines
and their reuse, not a general claim that backward chaining is always faster.
The app retains the latest time for each algorithm on the selected puzzle,
so both measurements can be compared side by side. Changing puzzles clears
the comparison; every solve still starts with a fresh KB.

## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


## Submission and deployed application

**Group:** Assignment and Project Group 22

The submitted folder contains only `Sudoku_Assignment.ipynb`, `sudoku_solver.py`
and `sudoku_app.py`. The original support files are unchanged and supplied separately
by the course. The deployment repository also needs those support files and
`requirements.txt` (including NumPy, which `utils.py` imports).

**Deployed Streamlit app URL:** [https://it5005-group22-sudoku-mcvpsg3aywdudne6ouc6hj.streamlit.app/](https://it5005-group22-sudoku-mcvpsg3aywdudne6ouc6hj.streamlit.app/)

**Source repository:** https://github.com/Takegiri/it5005-group22-sudoku

**Access:** The deployed app can be opened without signing in. Select a puzzle and run each chaining algorithm to compare timings. Use a cell query with tutor mode enabled to inspect the recorded proof.


## Additional correctness checks

These small exhaustive checks use classical truth-table semantics as an independent oracle. They verify both directions of the CNF encoding and test backward chaining even when rules or premises repeat. The Sudoku tests above separately compare all candidates against the supplied FC routine.

In [8]:
import itertools
import random

# Enumerate every Boolean interpretation of a 2x2 encoding. Exactly the
# two Latin squares satisfy it, with and without each possible given.
symbols = [atom('Is', r, c, v) for r in (1, 2) for c in (1, 2) for v in (1, 2)]
for givens in [{}, {(1, 1): 1}, {(1, 1): 2}]:
    kb = build_general_kb(2, 1, 2, givens)
    count = 0
    for bits in itertools.product([False, True], repeat=8):
        model = dict(zip(symbols, bits))
        cells = {(r,c): [v for v in (1,2) if model[atom('Is',r,c,v)]]
                 for r in (1,2) for c in (1,2)}
        legal = (all(len(vs)==1 for vs in cells.values()) and
                 all(cells[r,1] != cells[r,2] for r in (1,2)) and
                 all(cells[1,c] != cells[2,c] for c in (1,2)) and
                 all(cells[cell] == [v] for cell,v in givens.items()))
        result = all(pl_true(clause, model) for clause in kb.clauses)
        assert result == legal
        count += result
    assert count == (1 if givens else 2)
print('PASS: all 768 Boolean interpretations of tiny Sudoku encodings.')

# Use actual classical model enumeration as the oracle; keep duplicate
# rules and repeated premises, and ask queries in shuffled order.
rng = random.Random(225005)
atoms = [expr(f'T{i}') for i in range(5)]
for trial in range(300):
    kb = PropDefiniteKB()
    for a in atoms:
        if rng.random() < .2:
            kb.tell(a)
    for _ in range(12):
        premises = [rng.choice(atoms) for _ in range(rng.randint(1,3))]
        rule = Expr('==>', associate('&', premises), rng.choice(atoms))
        kb.tell(rule)
        if rng.random() < .15:
            kb.tell(rule)
    models = [dict(zip(atoms,bits)) for bits in itertools.product([False,True],repeat=5)]
    models = [m for m in models if all(pl_true(c,m) for c in kb.clauses)]
    assert models
    for q in rng.sample(atoms,5):
        assert pl_bc_entails(kb,q) == all(m[q] for m in models), (trial,q)
        proof = backward_proof(kb,q)
        seen = set()
        for head, premises in proof:
            assert set(premises) <= seen
            assert any(h == head and set(ps) == set(premises)
                       for ps,h in map(parse_definite_clause,kb.clauses))
            seen.add(head)
print('PASS: 1500 Horn queries against exhaustive truth tables, with cycles and duplicate premises/rules.')


PASS: all 768 Boolean interpretations of tiny Sudoku encodings.
PASS: 1500 Horn queries against exhaustive truth tables, with cycles and duplicate premises/rules.
